Imports

In [1]:
import pandas as pd

Load Data

In [2]:
finviz_data = pd.read_csv('Data/finviz_news.csv')
finnhub_data = pd.read_csv('Data/finnhub_news.csv')

Inspect Data

In [3]:
finviz_data.head()
finnhub_data.head()

,category,datetime,headline,id,image,related,source,summary,url,ticker_symbol,date
0,company,1774724400,"Inside ‘Project Eagle,’ JPMorgan’s High-Wire A...",139566024,https://s.yimg.com/rz/stage/p/yahoo_finance_en...,JPM,Yahoo,(Bloomberg) -- The post from @realDonaldTrump ...,https://finnhub.io/api/news?id=080a2fd22af214f...,JPM,2026-03-28 19:00:00
1,company,1774702800,"AI Schism Grips Washington as Tech, Labor Vie ...",139563802,https://s.yimg.com/rz/stage/p/yahoo_finance_en...,JPM,Yahoo,(Bloomberg) -- At a historic auditorium in Was...,https://finnhub.io/api/news?id=95a2b6b85c222cc...,JPM,2026-03-28 13:00:00
2,company,1774688452,Dip-buyers arrive to pull gold back from brink...,139563825,https://s.yimg.com/rz/stage/p/yahoo_finance_en...,JPM,Yahoo,(Bloomberg) -- Opportunistic buyers are starti...,https://finnhub.io/api/news?id=64539ff93e63ba3...,JPM,2026-03-28 09:00:52
3,company,1774656929,How The Standard Chartered (LSE:STAN) Story Is...,139559808,https://s.yimg.com/rz/stage/p/yahoo_finance_en...,JPM,Yahoo,Standard Chartered’s central fair value estima...,https://finnhub.io/api/news?id=2b15d000f555aa4...,JPM,2026-03-28 00:15:29
4,company,1774654553,Epstein victims to get $72.5M from Bank of Ame...,139565308,https://image.cnbcfm.com/api/v1/image/10825123...,JPM,CNBC,The settlement by Bank of America comes nearly...,https://finnhub.io/api/news?id=ff63226533fe0d9...,JPM,2026-03-27 23:35:53


Clean Data

In [4]:
import re
from ast import literal_eval

def parse_finviz_cell(raw_value):
    if pd.isna(raw_value):
        return pd.Series({'date': pd.NaT, 'headline': pd.NA})
    text = str(raw_value).strip()
    try:
        parsed = literal_eval(text)
        if isinstance(parsed, tuple) and len(parsed) >= 2:
            raw_date, raw_headline = parsed[0], parsed[1]
            return pd.Series({
                'date': pd.to_datetime(raw_date, errors='coerce'),
                'headline': str(raw_headline).strip() if raw_headline is not None else pd.NA
            })
    except Exception:
        pass
    date_match = re.search(r"\(\s*'([^']+)'\s*,", text)
    head_match = re.search(r"\(\s*'[^']+'\s*,\s*'(.+?)'\s*,\s*'https?://", text)
    fallback_date = pd.to_datetime(date_match.group(1), errors='coerce') if date_match else pd.NaT
    fallback_headline = head_match.group(1).strip() if head_match else pd.NA
    return pd.Series({'date': fallback_date, 'headline': fallback_headline})

finviz_long = finviz_data.melt(var_name='stock', value_name='raw_news').dropna(subset=['raw_news'])
parsed = finviz_long['raw_news'].apply(parse_finviz_cell)
finviz_clean = pd.concat([finviz_long[['stock']], parsed], axis=1)

finnhub_clean = finnhub_data.rename(columns={'ticker_symbol': 'stock'})[['stock', 'headline', 'date']].copy()
finnhub_clean['date'] = pd.to_datetime(finnhub_clean['date'], errors='coerce')
finnhub_clean['headline'] = finnhub_clean['headline'].astype(str).str.strip()

for df in (finviz_clean, finnhub_clean):
    df['stock'] = df['stock'].astype(str).str.upper().str.strip()
    df['headline'] = df['headline'].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

finviz_clean = finviz_clean.dropna(subset=['stock', 'headline', 'date']).drop_duplicates()
finviz_clean['date_only'] = finviz_clean['date'].dt.date
finnhub_clean = finnhub_clean.dropna(subset=['stock', 'headline', 'date']).drop_duplicates()
finnhub_clean['date_only'] = finnhub_clean['date'].dt.date

Save Cleaned Data

In [5]:
finviz_clean.to_csv('Data/finviz_clean.csv', index=False)
finnhub_clean.to_csv('Data/finnhub_clean.csv', index=False)